# _**Image segmentation - Azure deploy notebook**_

## Get segmentation model

In [7]:
import azureml.core
from azureml.core import Workspace, Dataset
import numpy as np
import tempfile
import keras

ws = Workspace.from_config()

ds_segim_name = 'ds_segim'

# model path
ds_segim = ws.datasets[ds_segim_name]
mounted_path = tempfile.mkdtemp()
mount_context = ds_segim.mount(mounted_path)
mount_context.start()
model_path = mounted_path + ds_segim.to_path()[0]

# load model
model = keras.models.load_model(model_path, compile=False)



Downloaded path: /tmp/tmp9u78gp9h/a3d89c47-a90b-4900-a742-4d28976fbf7a/UI/03-17-2022_124358_UTC/segim_model.h5 is different from target path: /tmp/tmp9u78gp9h/a3d89c47-a90b-4900-a742-4d28976fbf7a/segim_model.h5


## Register the model

In [8]:
from azureml.core.model import Model

model_registered = Model.register(ws, model_name="segim", model_path=model_path)


Registering model segim


## Create environment for the deploy

In [21]:
from azureml.core.environment import Environment
from azureml.core.conda_dependencies import CondaDependencies
from azureml.core.webservice import AciWebservice

env = Environment.from_conda_specification(name='dependencies_env', file_path="conda_dependencies.yml")

# create deployment config i.e. compute resources
aciconfig = AciWebservice.deploy_configuration(
    cpu_cores=1,
    memory_gb=1,
    tags={"data": "segmentation", "method": "unet"},
    description="Semantic segmentation with Unet",
)

## Deploy the model

In [22]:
%%time
import uuid
from azureml.core.model import InferenceConfig
from azureml.core.environment import Environment
from azureml.core.model import Model

# get the registered model
model_reg = Model(ws, "segim")

# create an inference config i.e. the scoring script and environment
inference_config = InferenceConfig(entry_script="score.py", environment=env)

# deploy the service
service_name = "segim-" + str(uuid.uuid4())[:4]
service = Model.deploy(
    workspace=ws,
    name=service_name,
    models=[model_reg],
    inference_config=inference_config,
    deployment_config=aciconfig,
)

service.wait_for_deployment(show_output=True)

Tips: You can try get_logs(): https://aka.ms/debugimage#dockerlog or local deployment: https://aka.ms/debugimage#debug-locally to debug if deployment takes longer than 10 minutes.
Running
2022-03-17 15:17:24+00:00 Creating Container Registry if not exists.
2022-03-17 15:17:24+00:00 Registering the environment.
2022-03-17 15:17:26+00:00 Use the existing image.
2022-03-17 15:17:26+00:00 Generating deployment configuration.
2022-03-17 15:17:28+00:00 Submitting deployment to compute..
2022-03-17 15:17:37+00:00 Checking the status of deployment segim-5ade..
2022-03-17 15:19:52+00:00 Checking the status of inference endpoint segim-5ade.
Succeeded
ACI service creation operation finished, operation "Succeeded"
CPU times: user 343 ms, sys: 63.3 ms, total: 407 ms
Wall time: 2min 49s


In [37]:
service.scoring_uri

'http://b9c28fdc-4242-44c0-bf03-2945b6a7e161.centralus.azurecontainer.io/score'